# TailRisk Solutions: Data-Driven Strategies for Financial Resilience in Energy Procurement
**Course:** 42578 Advanced Business Analytics | Technical University of Denmark (DTU)  
**Theme:** Intelligent Methods for Resilience  
**Group:** 17  
**Consultants:** [Insert Names + Student Number]

---

## Executive Summary
This project develops an advanced Decision Support System (DSS) for industrial manufacturers to mitigate financial exposure in the Iberian Electricity Market (MIBEL). Energy procurement is traditionally treated as a binary choice: either lock in long-term fixed contracts (sacrificing flexibility) or purchase daily on the Spot market (incurring catastrophic tail risks). 

**TailRisk Solutions** introduces a hybrid strategy: **Baseload Hedging combined with Operational Peak-Shaving**. 
Our framework splits the factory's energy demand into two layers:
1.  **Financial Resilience (Long-Term):** Securing the factory's "must-run" baseline production using OMIP Monthly Futures (M1 to M6), locking in costs and neutralizing extreme market spikes.
2.  **Operational Flexibility (Short-Term):** Leaving discretionary production capacity exposed to the Day-Ahead market. Utilizing a Reinforcement Learning agent, the DSS evaluates tomorrow's deterministic price ($t+1$) against the stochastic forecasts for the following 48-72 hours ($t+2$ and $t+3$). The agent dynamically shifts this discretionary production—overproducing during cheap renewable energy gluts and halting discretionary lines during price peaks.

This report details the end-to-end pipeline, from integrating heterogeneous financial and meteorological data streams to engineering market-spread features, culminating in a resilient, dual-action Reinforcement Learning environment.

---

## Table of Contents
1.  **Section 0: Environment Setup & Dependencies**
2.  **Section 1: Business Context & The Dual-Layer Strategy**
    * 1.1 The MIBEL Market Mechanics
    * 1.2 Layer 1: Baseload Hedging via Futures
    * 1.3 Layer 2: The $t+1$ Fixed Horizon & Production Shifting
3.  **Section 2: Data Engineering Pipeline**
    * 2.1 Multi-Source Extraction (Financial & Multi-Batch Weather)
    * 2.2 Exploratory Data Analysis (Volatility & Cointegration)
    * 2.3 Technical Cleaning & Temporal Alignment
4.  **Section 3: Advanced Feature Engineering**
    * 3.1 Financial Spreads (Spot vs. M1) & Momentum
    * 3.2 Target Engineering: $t+2$ and $t+3$ Foresight Signals
5.  **Section 4: Feature Selection & The "Anti-Leakage Shield"**
6.  **Section 5: The Decision Engine (Reinforcement Learning)**
7.  **Section 6: Counterfactual Backtesting & Results**
8.  **Section 7: Strategic Recommendations**

---

## 0. Environment Setup & Dependencies

### 0.1 Prerequisites
To ensure reproducibility, install the necessary dependencies using the `requirements.txt` file located in the project root:

```bash
pip install -r ../requirements.txt


### 0.2 Master Dependencies & Environment Setup
To ensure the academic rigor and reproducibility required by DTU standards, all dependencies are centralized. This manifesto includes tools for high-performance data manipulation, statistical visualization, and the machine learning frameworks used for risk quantification.

In [19]:
# ==============================================================================
# SECTION 0.2: MASTER DEPENDENCIES & GLOBAL CONFIGURATION
# ==============================================================================

# 1. System & Operations
import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

# 2. Data Manipulation & Numerical Computing
import numpy as np
import pandas as pd

# 3. Visualization Suite (Academic Standard)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go # For interactive financial charts

# 4. Machine Learning & Statistical Tools
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import mutual_info_regression
from scipy.ndimage import gaussian_filter1d 

# 5. Reinforcement Learning Foundations
# import gymnasium as gym
# from stable_baselines3 import PPO, DQN

# Environment Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid') # Professional plotting style
%matplotlib inline

# Global Pandas Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Project Constants (Standardized across the pipeline)
TARGET_HORIZONS = [2, 3]  # Optimizing daily shifts for t+2 and t+3
HEDGE_TARGET = 'Future_M1'  # Primary instrument for baseload risk mitigation
RANDOM_SEED = 42    # For reproducibility
print(f"✅ TailRisk Solutions Environment Loaded | Python {sys.version.split()[0]}")

✅ TailRisk Solutions Environment Loaded | Python 3.12.11


---

## 1. Business Context & The Dual-Layer Strategy

### 1.1 The Market Architecture
The Iberian Electricity Market operates across two primary timeframes, both of which are required to build a resilient procurement strategy:
* **OMIP (Derivatives Market):** Where financial futures (Months M1 through M6) are traded. These instruments reflect the macroeconomic expectations of the market and allow consumers to lock in prices months in advance.
* **OMIE (Day-Ahead Spot Market):** A blind auction clearing at 13:00 CET on day $t$ for all 24 hours of day $t+1$. This market is highly volatile, driven by immediate weather conditions (wind/solar generation) and geopolitical gas shocks.

### 1.2 Layer 1: Baseload Risk Mitigation (The Hedge)
To prevent operational bankruptcy during sustained energy crises, our strategy establishes a **Baseload Hedge**. A predetermined percentage of the factory's minimum required energy is purchased via OMIP Futures (e.g., locking in a fixed rate for `Future_M1`). This guarantees that the core production line operates at a known, stable cost, immunizing the business against extreme tail risks.

### 1.3 Layer 2: Discretionary Production Shifting (The Spot Advantage)
While hedging mitigates risk, it eliminates the opportunity to capitalize on zero-price days driven by renewable overproduction. The remaining, unhedged percentage of our energy demand is managed dynamically by our Decision Support System.

Operating under the OMIE temporal constraints, at 13:00 CET on day $t$, the cost for tomorrow ($t+1$) becomes deterministic. Our RL agent compares this fixed $t+1$ cost against the stochastic forecasts for $t+2$ and $t+3$. 
* **Action - Overproduce:** If $t+1$ is cheap relative to $t+2/t+3$, the factory increases its discretionary production, storing excess inventory using cheap Spot energy.
* **Action - Curtail:** If $t+1$ is expensive, the factory shuts down discretionary lines, relying on inventory and waiting for the cheaper energy forecasted in the coming days.

By treating the factory as a flexible asset acting on the spread between long-term hedges and short-term spot volatility, we achieve true financial resilience.

### 1.4 Modeling Assumption: Daily Baseload Price
To align the analytics with macro-level industrial planning and reduce state-space complexity for the Reinforcement Learning agent, we assume a **Constant Daily Price**. We utilize the daily average (baseload) price as the primary economic signal for the Spot market. This focuses the optimization on the "Day-to-Day" shifting logic rather than intra-day hourly volatility, which often averages out over a full production shift.

---

## 2. Data Engineering Pipeline: Ingestion Strategies

The robustness of our Decision Support System (DSS) depends fundamentally on the reliability of its data streams. This section details the "Web Data Mining" strategies employed to build our dataset, highlighting how we engineered the ingestion pipelines to be inherently resilient to network failures and strict rate limits.

### 2.1 Financial Data Acquisition: The Web Scraping Imperative
Unlike standard financial markets, the **OMIP (Iberian Energy Derivatives Exchange)** does not expose a public API for historical settlement data. To overcome this information asymmetry, we developed an automated **Web Scraper** (`src/extraction/extract_omip.py`).
* **Extraction Logic:** The scraper programmatically navigates the OMIP HTML Document Object Model (DOM) to extract daily closing prices for **Monthly Futures (M1 to M6)** and **Open Interest** levels.
* **Analytical Value:** Open Interest acts as a proxy for market liquidity and collective sentiment. Capturing this "dark data" provides our Reinforcement Learning agent with foresight regarding shifts in market positioning that are invisible to price-only models.

### 2.2 Meteorological Stream: API Architecture
While financial data reflects market expectations, weather data dictates the physical reality of energy supply. To capture this, we engineered a highly fault-tolerant ingestion engine (`src/extraction/extract_weather.py`) interacting with the **Open-Meteo Archive API**.

To extract daily historical data (2020-2025) across 52 Spanish provinces without triggering server bans or losing data due to timeouts, we implemented several defensive programming paradigms:
1.  **Geospatial Batching:** Province coordinates (latitude/longitude) are loaded from an external metadata file (`_Provincias_Info.xlsx`) and partitioned into controlled batches (e.g., indices 0-10, 11-51) to strictly respect API payload limits.
2.  **Network Resilience (Caching & Retries):** We integrated `requests_cache` to locally store API responses, preventing redundant network calls during development. Furthermore, we wrapped the HTTP session in a `retry_requests` adapter with an exponential backoff factor. If the Open-Meteo server drops a connection, our pipeline automatically waits and safely reattempts the download, ensuring absolute data completeness.
3.  **National Aggregation:** In downstream processing, these 52 regional time series are weighted and aggregated into a "National Peninsular Index." This smooths localized sensor noise while amplifying macro-climatic events (e.g., severe wind fronts) that directly collapse Iberian spot prices.

In [20]:
# ==============================================================================
# SECTION 2.1 & 2.2: MODULAR DATA INGESTION & VALIDATION
# ==============================================================================

# Path resolution for modular imports
if str(Path.cwd().resolve().parents[1]) not in sys.path:
    sys.path.append(str(Path.cwd().resolve().parents[1]))

try:
    from src.data.load_raw_data import load_raw_data
    print("✅ Modular loaders successfully imported.")
except ImportError as e:
    print(f"❌ Critical Import Error: {e}")
    raise

# Execute the Ingestion Pipeline
try:
    raw_datasets = load_raw_data()
    
    # Store in explicit variables for the analytical pipeline
    df_omip_raw = raw_datasets['omip']
    df_weather_raw = raw_datasets['weather']
    df_holidays_raw = raw_datasets['holidays']
    
    print(f"\n📊 Ingestion Audit Passed:")
    print(f"   - Financial (OMIP): {df_omip_raw.shape[0]} days captured.")
    print(f"   - Weather (API):    {df_weather_raw.shape[0]} days captured.")
    print(f"   - Holidays:         {df_holidays_raw.shape[0]} days captured.")
    
except Exception as e:
    print(f"❌ Pipeline Execution Failed: {e}")
    raise

✅ Modular loaders successfully imported.

📊 Ingestion Audit Passed:
   - Financial (OMIP): 2192 days captured.
   - Weather (API):    2275 days captured.
   - Holidays:         2192 days captured.
